In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generate flow that highly depends on yesterday's flow (rho=0.8)
import scipy.signal

np.random.seed(42)
T = 100
rainfall = np.random.uniform(0, 20, T)
runoff_coeff = 0.5

# Prepare today's input (rainfall effect + random noise).
noise = np.random.normal(0, 1, T)
exog = runoff_coeff * rainfall + noise
exog[0] = 0  # Anchor first day so we match the base flow cleanly

# lfilter pushes the recursion to C: y[t] = 0.8*y[t-1] + exog[t]
# The 'a' array defines the left side of the equation: y[t] - 0.8*y[t-1] = exog[t]
flow_centered = scipy.signal.lfilter(b=[1.0], a=[1.0, -0.8], x=exog)
flow_true = flow_centered + 10  # Shift back up to resting baseline (10)

with pm.Model() as model_static:

    b_rain = pm.HalfNormal("b_rain", sigma=2)
    intercept = pm.Normal("intercept", mu=10, sigma=5)
    sigma = pm.HalfNormal("sigma", sigma=5)
    mu = pm.Deterministic("mu", intercept + b_rain * rainfall)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=flow_true)
    trace_static = pm.sample(1000, tune=1000, cores=1, random_seed=42)

    prior_checks = pm.sample_prior_predictive(draws=1000, random_seed=42)

    trace_static = pm.sample(draws=2000, tune=1000, chains=2, random_seed=42, progressbar=False)

static_preds = trace_static.posterior["mu"].mean(dim=["chain", "draw"])
residuals = flow_true - static_preds

plt.acorr(residuals, maxlags=20)
plt.title("M2 Static Model Residuals: MASSIVE AUTOCORRELATION")
plt.show()

In [ ]:
# The Invariant: Water takes time to leave the system. Yesterday's flow dictates today's baseline.
# Financial Liability: Failing to model autoregression (AR) in hydrology means underestimating
# the duration of flood events, leading to catastrophic under-design of diversion tunnels.

# Mean-center continuous data to eliminate the intercept trap.
# This decouples the baseline magnitude from the variance and AR coefficients.
flow_centered = flow_true - flow_true.mean()
rain_centered = rainfall - rainfall.mean()

with pm.Model() as m3_arx:
    #  We expect positive autocorrelation (water doesn't instantly vanish)
    rho = pm.Normal("rho", mu=0.5, sigma=0.5)

    # Rainfall translates directly to flow, but we leave the prior broad to let data speak
    b_rain = pm.Normal("b_rain", mu=0, sigma=2)

    # System noise: What variations in flow are not explained by rain or yesterday's flow?
    sigma = pm.HalfNormal("sigma", sigma=5)

    # The Exogenous forcing factor for the sequence
    mu_exo = b_rain * rain_centered

    # PyMC variables cannot be passed to 'observed' in pm.AR.
    # ARX relationship:
    # E[y_t] = rho * y_{t-1} + beta * X_t

    mu = pm.Deterministic("mu", rho * flow_centered[:-1] + mu_exo[1:])


    y = pm.Normal("y",
                  mu=mu,
                  sigma=sigma,
                  observed=flow_centered[1:])

    trace_arx = pm.sample(draws=1000,
                          tune=1000,
                          cores=1,
                          target_accept=0.95,
                          random_seed=42,
                          progressbar=False)


print(az.summary(trace_arx, var_names=["rho", "b_rain"]))